# 8 · Delta Lake — deep dive

**Delta Lake** is the open table format under every table on Databricks. It's
**Parquet files + a transaction log** (`_delta_log`) that together give you
warehouse-grade guarantees on cheap lake storage: **ACID transactions**,
**updates/deletes/MERGE**, **time travel**, **schema enforcement & evolution**,
and performance features like **OPTIMIZE** and **liquid clustering**.

This notebook covers the Delta features a data engineer uses daily, on the shared
**BrewBox** data.

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
from pyspark.sql import functions as F
spark.sql("USE SCHEMA brewbox")
print("Ready. Using schema: brewbox")

## 1 · A Delta table is a log

Every write appends a **commit** to the transaction log. That log is what makes
reads consistent (readers never see a half-written table) and enables time
travel. Let's make a small Delta table from the customers reference data and look
at its history.

In [ ]:
spark.sql("CREATE OR REPLACE TABLE brewbox.customers_dim AS SELECT * FROM brewbox.customers")
spark.sql("DESCRIBE HISTORY brewbox.customers_dim").select("version","operation","operationParameters").show(truncate=False)

## 2 · UPDATE, DELETE, and MERGE

Unlike plain Parquet, Delta supports row-level `UPDATE`/`DELETE` and the
all-important **`MERGE`** (upsert). `MERGE` matches a source against a target and
inserts new rows / updates changed ones in one atomic step — the backbone of
loading changes (CDC).

In [ ]:
# UPDATE + DELETE
spark.sql("UPDATE brewbox.customers_dim SET loyalty_tier = 'gold' WHERE customer_id = 1")
spark.sql("DELETE FROM brewbox.customers_dim WHERE customer_id = 40")
print("rows after update/delete:", spark.table("brewbox.customers_dim").count())

In [ ]:
# MERGE upsert: a source with one changed customer + one brand-new customer.
# We use the DeltaTable API so we can MERGE a Python DataFrame directly.
updates = spark.createDataFrame(
    [(1, "Ava Smith", "CA", "ava.smith@example.com", "2023-02-01", "gold"),      # moved US->CA
     (99, "New Person", "US", "new@example.com", "2024-07-01", "bronze")],        # brand new
    ["customer_id","name","country","email","signup_date","loyalty_tier"])

from delta.tables import DeltaTable
tgt = DeltaTable.forName(spark, "brewbox.customers_dim")
(tgt.alias("t").merge(updates.alias("s"), "t.customer_id = s.customer_id")
    .whenMatchedUpdate(set={"country": "s.country"})
    .whenNotMatchedInsertAll()
    .execute())

spark.table("brewbox.customers_dim").filter("customer_id IN (1, 99)").show()

## 3 · Slowly Changing Dimension (SCD) Type 2 with MERGE

SCD Type 2 keeps **history**: when a tracked attribute changes, you *close* the
old row and *insert* a new current one, so you can see what was true at any point.
The pattern is a `MERGE` that, on a match with a changed value, updates the old
row's `is_current`/`end_date` and inserts the new version.

In [ ]:
# Build an SCD2 dimension with validity columns
spark.sql("""
  CREATE OR REPLACE TABLE brewbox.dim_customer_scd2 AS
  SELECT customer_id, country, loyalty_tier,
         current_timestamp() AS effective_from,
         CAST(NULL AS TIMESTAMP) AS effective_to,
         true AS is_current
  FROM brewbox.customers
""")

# A change arrives: customer 2's country changes. Close the old, insert the new.
changes = spark.createDataFrame([(2, "FR", "silver")], ["customer_id","country","loyalty_tier"])
from delta.tables import DeltaTable
dim = DeltaTable.forName(spark, "brewbox.dim_customer_scd2")

# Step 1: expire current rows that changed
(dim.alias("t").merge(
    changes.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = true AND t.country <> s.country")
  .whenMatchedUpdate(set={"is_current": "false", "effective_to": "current_timestamp()"})
  .execute())

# Step 2: insert the new current version
new_versions = changes.selectExpr(
    "customer_id","country","loyalty_tier",
    "current_timestamp() AS effective_from",
    "CAST(NULL AS TIMESTAMP) AS effective_to","true AS is_current")
new_versions.write.mode("append").saveAsTable("brewbox.dim_customer_scd2")

spark.table("brewbox.dim_customer_scd2").filter("customer_id = 2").orderBy("effective_from").show(truncate=False)

## 4 · Time travel & RESTORE

Every version is queryable. Read an old version with `VERSION AS OF` or
`TIMESTAMP AS OF`, and **`RESTORE`** to roll the whole table back — a lifesaver
after a bad load.

In [ ]:
spark.sql("DESCRIBE HISTORY brewbox.customers_dim").select("version","operation").show()
print("current rows:", spark.table("brewbox.customers_dim").count())
print("rows at version 0:", spark.sql("SELECT count(*) c FROM brewbox.customers_dim VERSION AS OF 0").first()["c"])
# Roll the table back to version 0:
# spark.sql("RESTORE TABLE brewbox.customers_dim TO VERSION AS OF 0")

## 5 · Schema enforcement & evolution

Delta **enforces** schema by default (a mismatched write fails — protecting you
from bad data). When you *intend* to change the schema, opt in with
`mergeSchema` on write or `ALTER TABLE ... ADD COLUMN`.

In [ ]:
spark.sql("ALTER TABLE brewbox.customers_dim ADD COLUMN (marketing_opt_in BOOLEAN)")
spark.sql("UPDATE brewbox.customers_dim SET marketing_opt_in = true WHERE loyalty_tier = 'gold'")
spark.table("brewbox.customers_dim").select("customer_id","loyalty_tier","marketing_opt_in").show(5)

## 6 · Constraints & generated columns

Delta can enforce **`NOT NULL`** and **`CHECK`** constraints (bad rows are
rejected at write time), and compute **generated columns** automatically —
data-quality guarantees baked into the table.

In [ ]:
spark.sql("ALTER TABLE brewbox.customers_dim ALTER COLUMN customer_id SET NOT NULL")
spark.sql("ALTER TABLE brewbox.customers_dim ADD CONSTRAINT valid_tier CHECK (loyalty_tier IN ('bronze','silver','gold'))")
print("Constraints now protect the table (bad inserts will be rejected).")
spark.sql("DESCRIBE DETAIL brewbox.customers_dim").select("numFiles","sizeInBytes").show()

## 7 · Performance: OPTIMIZE, Z-order, liquid clustering, VACUUM

- **`OPTIMIZE`** compacts many small files into fewer large ones (the "small-file
  problem"). Add **`ZORDER BY (col)`** to co-locate related data so queries skip
  more files.
- **Liquid clustering** (`CLUSTER BY`) is the newer, automatic alternative to
  partitioning + Z-order — you set clustering keys and Databricks maintains them.
- **`VACUUM`** removes old, unreferenced files after the time-travel retention
  window (default 7 days) to save storage.

In [ ]:
spark.sql("OPTIMIZE brewbox.customers_dim ZORDER BY (country)")
# Liquid clustering alternative (on CREATE): CREATE TABLE ... CLUSTER BY (country)
# VACUUM (respects retention; here just show the command form):
# spark.sql("VACUUM brewbox.customers_dim RETAIN 168 HOURS")
spark.sql("DESCRIBE HISTORY brewbox.customers_dim").select("version","operation").show(5)

## 8 · Change Data Feed (CDF)

Enable **Change Data Feed** and Delta records every row-level change
(insert/update/delete). Downstream jobs can then read *just what changed* with
`table_changes(...)` — perfect for incremental Silver/Gold updates.

In [ ]:
spark.sql("ALTER TABLE brewbox.customers_dim SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql("UPDATE brewbox.customers_dim SET loyalty_tier = 'silver' WHERE customer_id = 3")

# Read the changes since the version where CDF was enabled onward
latest = spark.sql("SELECT max(version) v FROM (DESCRIBE HISTORY brewbox.customers_dim)").first()["v"]
spark.sql(f"SELECT customer_id, loyalty_tier, _change_type, _commit_version "
          f"FROM table_changes('brewbox.customers_dim', {latest}) LIMIT 20").show()

## 9 · Exercises

**Exercise 1 —** Show the full version history of `brewbox.customers_dim` with the
operation and the number of output rows (`operationMetrics`).

In [ ]:
# Your turn (Exercise 1):

In [ ]:
# ✅ Solution 1
spark.sql("DESCRIBE HISTORY brewbox.customers_dim").select(
    "version","operation","operationMetrics").show(truncate=False)

**Exercise 2 —** Use time travel to count how many rows `brewbox.customers_dim`
had at **version 0** vs now.

In [ ]:
# Your turn (Exercise 2):

In [ ]:
# ✅ Solution 2
v0 = spark.sql("SELECT count(*) c FROM brewbox.customers_dim VERSION AS OF 0").first()["c"]
now = spark.table("brewbox.customers_dim").count()
print("version 0:", v0, "| now:", now)

**Exercise 3 —** MERGE a new customer (id 500) into `brewbox.customers_dim`; if it
already exists update its `loyalty_tier`, else insert it.

In [ ]:
# Your turn (Exercise 3):

In [ ]:
# ✅ Solution 3
from delta.tables import DeltaTable
src = spark.createDataFrame([(500,"Test User","US","t@x.com","2024-01-01","silver",True)],
    ["customer_id","name","country","email","signup_date","loyalty_tier","marketing_opt_in"])
(DeltaTable.forName(spark, "brewbox.customers_dim").alias("t")
    .merge(src.alias("s"), "t.customer_id = s.customer_id")
    .whenMatchedUpdate(set={"loyalty_tier": "s.loyalty_tier"})
    .whenNotMatchedInsertAll().execute())
spark.table("brewbox.customers_dim").filter("customer_id = 500").show()

## 10 · Recap & next

Delta gives you ACID, `MERGE`/SCD, time travel + `RESTORE`, schema
enforcement/evolution, constraints, `OPTIMIZE`/Z-order/liquid clustering,
`VACUUM`, and Change Data Feed — warehouse power on lake storage.

**Next → `9` Transformations:** clean and enrich the raw **Bronze** orders into a
trustworthy **Silver** table using PySpark and Spark SQL. 🚀